# Online Course Recommendation System — Data Cleaning & EDA
**Project:** P683 — Online Course Recommendation System Dataset V2

**Milestone:** Data Cleaning + EDA

This notebook covers:
1. Loading and first look at the data
2. Data cleaning (missing values, duplicates, range checks, outliers, consistency checks)
3. Exploratory Data Analysis (univariate, bivariate, correlation)
4. Saving the cleaned dataset for the next milestone (Model Building)


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


## 2. Load the Dataset

In [ ]:
# Update this path if your file is located elsewhere
FILE_PATH = "online_course_recommendation_v2.xlsx"

df = pd.read_excel(FILE_PATH)
print("Shape (rows, columns):", df.shape)
df.head()


## 3. First Look at the Data

Before cleaning anything, we check the structure: column names, data types,
and basic statistics. This tells us what kind of cleaning is actually needed
instead of guessing.

In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


In [ ]:
# Column-wise data types — useful to confirm categorical vs numeric vs id columns
df.dtypes


## 4. Missing Values

Check every column for nulls. Even if a column shows 0 missing values,
it's worth checking for "hidden" missing values like empty strings or
placeholder text (e.g. "NA", "Unknown", "-").

In [ ]:
# Standard missing value check
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df[missing_df.missing_count > 0].sort_values('missing_count', ascending=False)


In [ ]:
# Hidden missing values disguised as strings (common in real-world data)
placeholder_values = ['NA', 'N/A', 'na', 'n/a', 'null', 'NULL', '-', '', ' ', 'Unknown', 'unknown']
string_cols = [c for c in df.columns if df[c].dtype == object or df[c].dtype.name == 'str']

for col in string_cols:
    found = df[col].isin(placeholder_values).sum()
    if found > 0:
        print(f"{col}: {found} placeholder/hidden-missing values found")
print("Check complete — if nothing printed above, no hidden missing values were found.")


**Cleaning action:** If missing values are found, handle them based on column type:
- Numeric columns → impute with median (robust to outliers) or drop rows if very few
- Categorical columns → impute with mode, or a new category like `'Unknown'`
- If a column has a very high % missing (e.g. >40%), consider dropping the column

Uncomment and adapt the cell below if your run of the dataset has missing values.

In [ ]:
# Example imputation pattern (uncomment & adapt to the columns that actually need it)

# numeric_cols = ['course_duration_hours', 'rating', 'enrollment_numbers', 'course_price',
#                  'feedback_score', 'time_spent_hours', 'previous_courses_taken']
# for col in numeric_cols:
#     if df[col].isnull().sum() > 0:
#         df[col] = df[col].fillna(df[col].median())

# categorical_cols = ['certification_offered', 'difficulty_level', 'study_material_available']
# for col in categorical_cols:
#     if df[col].isnull().sum() > 0:
#         df[col] = df[col].fillna(df[col].mode()[0])


## 5. Duplicate Records

Two checks:
1. Fully duplicated rows (every column identical)
2. Duplicate `(user_id, course_id)` pairs — the same learner enrolled in the
   same course more than once. Depending on what the data is supposed to
   represent, this may be a genuine repeat enrollment or a data entry error.

In [ ]:
full_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {full_dupes}")

if full_dupes > 0:
    df = df.drop_duplicates()
    print(f"Dropped. New shape: {df.shape}")


In [ ]:
full_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {full_dupes}")

if full_dupes > 0:
    df = df.drop_duplicates()
    print(f"Dropped. New shape: {df.shape}")


**Cleaning action:** For a recommendation system, a user normally appears once
per course (one rating/interaction per user-course pair). We keep the **first**
occurrence and drop the rest — adjust this rule if your use case treats repeat
enrollments as meaningful (e.g. retakes).

In [ ]:
df = df.drop_duplicates(subset=['user_id', 'course_id'], keep='first')
print("Shape after removing duplicate user-course pairs:", df.shape)


## 6. Range & Consistency Checks

The business document specifies expected ranges for several columns. We
validate that the data actually respects them, and check for negative
values where they shouldn't exist.

In [ ]:
range_rules = {
    'course_duration_hours': (5.0, 100.0),
    'rating': (1.0, 5.0),
    'course_price': (20.0, 500.0),
    'feedback_score': (0.0, 1.0),
    'time_spent_hours': (1.0, 100.0),
}

for col, (low, high) in range_rules.items():
    out_of_range = df[(df[col] < low) | (df[col] > high)]
    print(f"{col}: {len(out_of_range)} rows outside [{low}, {high}]")


In [ ]:
# Columns that should never be negative
non_negative_cols = ['enrollment_numbers', 'previous_courses_taken', 'user_id', 'course_id']
for col in non_negative_cols:
    neg = (df[col] < 0).sum()
    print(f"{col}: {neg} negative values")


In [ ]:
# Categorical values should match exactly what's documented (Yes/No, Beginner/Intermediate/Advanced)
for col in ['certification_offered', 'study_material_available']:
    print(col, '->', sorted(df[col].unique()))

print('difficulty_level ->', sorted(df['difficulty_level'].unique()))


In [ ]:
# Consistency check: does each course_name always map to the same instructor & difficulty?
# (a course shouldn't randomly change instructor across rows)
inconsistent_instructor = df.groupby('course_name')['instructor'].nunique()
inconsistent_instructor = inconsistent_instructor[inconsistent_instructor > 1]
print("Courses with more than one instructor on record:", len(inconsistent_instructor))
inconsistent_instructor


In [ ]:
# Whitespace / casing issues in text columns (e.g. "Yes " vs "Yes", "yes" vs "Yes")
text_cols = ['course_name', 'instructor', 'certification_offered', 'difficulty_level', 'study_material_available']
for col in text_cols:
    has_ws = (df[col].astype(str) != df[col].astype(str).str.strip()).sum()
    print(f"{col}: {has_ws} values with leading/trailing whitespace")

# Standardize just in case
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()


## 7. Outlier Detection (IQR Method)

For numeric columns without a hard documented range (e.g. `enrollment_numbers`,
`previous_courses_taken`), we use the IQR (Interquartile Range) method to flag
statistical outliers. We flag them here rather than blindly deleting them —
outliers in enrollment counts may be genuinely popular courses, not errors.

In [ ]:
numeric_cols = ['course_duration_hours', 'rating', 'enrollment_numbers', 'course_price',
                 'feedback_score', 'time_spent_hours', 'previous_courses_taken']

def iqr_outlier_summary(data, cols):
    rows = []
    for col in cols:
        q1, q3 = data[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outliers = data[(data[col] < lower) | (data[col] > upper)]
        rows.append({
            'column': col, 'lower_bound': round(lower, 2), 'upper_bound': round(upper, 2),
            'outlier_count': len(outliers), 'outlier_pct': round(100 * len(outliers) / len(data), 2)
        })
    return pd.DataFrame(rows)

outlier_summary = iqr_outlier_summary(df, numeric_cols)
outlier_summary


In [ ]:
# Visualize outliers with boxplots
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='steelblue')
    axes[i].set_title(col)
axes[-1].axis('off')
plt.tight_layout()
plt.show()


**Cleaning action:** Since `enrollment_numbers` and similar columns can be
legitimately skewed (a few very popular courses), we **cap (winsorize)** rather
than delete outliers, preserving row count for the recommendation model. Adjust
this if your assignment asks for outlier removal instead.

In [ ]:
# Capping (winsorizing) instead of deleting — keeps all rows for the recommender system
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df[col] = df[col].clip(lower=lower, upper=upper)

print("Outliers capped. Re-checking:")
iqr_outlier_summary(df, numeric_cols)


## 8. Save the Cleaned Dataset

In [ ]:
df.to_csv("course_recommendation_cleaned.csv", index=False)
print("Cleaned dataset saved as 'course_recommendation_cleaned.csv'")
print("Final shape:", df.shape)


## 9. Exploratory Data Analysis (EDA)

### 9.1 Univariate Analysis — Numeric Columns

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Distribution of {col}')
axes[-1].axis('off')
plt.tight_layout()
plt.show()


### 9.2 Univariate Analysis — Categorical Columns

In [ ]:
categorical_cols = ['certification_offered', 'difficulty_level', 'study_material_available']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(categorical_cols):
    sns.countplot(x=df[col], hue=df[col], ax=axes[i], palette='viridis', legend=False)
    axes[i].set_title(f'Count of {col}')
plt.tight_layout()
plt.show()


In [ ]:
# Most popular courses by enrollment
top_courses = df.groupby('course_name')['enrollment_numbers'].sum().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_courses.values, y=top_courses.index, hue=top_courses.index, palette='mako', legend=False)
plt.title('Total Enrollment by Course')
plt.xlabel('Total Enrollments')
plt.show()


### 9.3 Bivariate Analysis

In [ ]:
# Correlation heatmap between numeric features
plt.figure(figsize=(10, 7))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Features')
plt.show()


In [ ]:
# Does difficulty level relate to rating?
plt.figure(figsize=(8, 5))
sns.boxplot(x='difficulty_level', y='rating', hue='difficulty_level', data=df, order=['Beginner', 'Intermediate', 'Advanced'], palette='Set2', legend=False)
plt.title('Rating by Difficulty Level')
plt.show()


In [ ]:
# Does certification affect course price or enrollment?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='certification_offered', y='course_price', hue='certification_offered', data=df, ax=axes[0], palette='Set3', legend=False)
axes[0].set_title('Course Price by Certification Offered')
sns.boxplot(x='certification_offered', y='enrollment_numbers', hue='certification_offered', data=df, ax=axes[1], palette='Set3', legend=False)
axes[1].set_title('Enrollment by Certification Offered')
plt.tight_layout()
plt.show()


In [ ]:
# Relationship between time spent and rating — engaged learners may rate differently
plt.figure(figsize=(8, 5))
sns.scatterplot(x='time_spent_hours', y='rating', data=df, alpha=0.3, color='darkorange')
plt.title('Time Spent vs Rating')
plt.show()


In [ ]:
# Does previous learning history relate to engagement (time spent)?
plt.figure(figsize=(8, 5))
sns.scatterplot(x='previous_courses_taken', y='time_spent_hours', data=df, alpha=0.3, color='purple')
plt.title('Previous Courses Taken vs Time Spent')
plt.show()


### 9.4 Course-Level Summary Table

In [ ]:
course_summary = df.groupby('course_name').agg(
    avg_rating=('rating', 'mean'),
    total_enrollments=('enrollment_numbers', 'sum'),
    avg_price=('course_price', 'mean'),
    avg_feedback_score=('feedback_score', 'mean'),
    avg_time_spent=('time_spent_hours', 'mean')
).round(2).sort_values('avg_rating', ascending=False)

course_summary


## 10. Summary

- Checked and handled missing values (none found in the original file, but the
  notebook includes the logic in case your copy differs)
- Removed fully duplicate rows and duplicate `(user_id, course_id)` pairs
- Validated all numeric columns against the documented business ranges
- Checked categorical consistency (whitespace, casing, instructor-course mapping)
- Detected and capped statistical outliers using the IQR method
- Saved the cleaned dataset to `course_recommendation_cleaned.csv`
- Explored distributions, correlations, and course-level patterns to inform
  the next milestone: **Model Building & Evaluation** (collaborative /
  content-based / hybrid recommendation approaches)
